# ECGR 4161 — Particle Filter Robot Localization

## Overview

In the last two labs you localized a receiver by **solving equations**: UWB
trilateration intersected distance circles, and GPS used **least squares** to
fold in a fourth unknown (the clock bias). Both assume the world is close enough
to a single, well-behaved solution that calculus can march you straight to it.

Real robots are messier. Motion is uncertain, sensors are noisy, and sometimes
the robot has **no idea where it starts** — it could be anywhere in the room.
When the belief about "where am I?" is not a tidy bell curve, a single
best-fit point is the wrong tool.

The **particle filter** (also called **Monte Carlo Localization**, MCL) takes a
completely different stance. Instead of tracking one estimate, it represents the
robot's belief with a **cloud of hundreds of guesses** — the *particles*. Each
particle is a hypothesis "maybe the robot is *here*, facing *this* way." Every
time step the swarm:

1. **moves** the way the robot thinks it moved (with a little randomness),
2. **scores** each particle by how well its predicted sensor readings match the
   real measurement,
3. **keeps the good guesses and discards the bad ones** (resampling), and
4. **jitters the survivors** so the swarm can keep exploring.

Run this loop a few times and the cloud collapses onto the true pose — even if
it started spread across the entire arena.

| | GPS least squares (Lab 5) | Particle filter (this lab) |
|---|---|---|
| Belief representation | One estimate $[x,y,z,b]$ | A cloud of $N$ weighted particles |
| Works when belief is | A single Gaussian blob | **Any** shape (multi-modal, uniform, …) |
| Needs a good initial guess? | Helps, but converges from center | **No** — can start totally lost |
| Core operation | Linearize + solve normal equations | Sample, weight, **resample**, jitter |
| Cost | Cheap (one small solve) | Grows with the number of particles |

## The robot and its world

A wheeled robot drives a curved path around a square arena. Bolted to the walls
are a handful of **landmark beacons** at known locations (just like UWB
anchors). Each time step the robot:

- receives a **control** — a forward speed $v$ and a turn rate $\omega$ — which
  it uses to guess how it moved (its odometry is imperfect), and
- measures the **distance to every landmark** (noisy range readings).

Your particle filter has to fuse the uncertain motion and the noisy ranges into
a good estimate of the robot's pose $[x, y, \theta]$.

## Your job

You will implement the **three knobs that make a particle filter work** and then
**experiment** with them:

1. `initialize_particles(...)` — how the initial cloud is drawn (uniform across
   the whole arena, a Gaussian around a guess, or a single collapsed point).
2. `resample(...)` — how the good particles are kept and the bad ones dropped.
3. `roughen(...)` — how noise is sprinkled onto the survivors so the swarm does
   not collapse to a lifeless dot.

The motion model, the measurement model, the simulation, and the animated
visualization are all provided.

## Learning Objectives

After completing this notebook, you should be able to:

1. Describe the **predict → weight → resample → roughen** cycle of a particle
   filter and how it differs from a least-squares estimator.
2. Explain how the **number of particles** trades accuracy and robustness
   against computational cost.
3. Compare **initial particle distributions** (uniform vs. Gaussian vs. a single
   point) and reason about which to use when the robot is lost vs. roughly
   known.
4. Explain **particle deprivation / sample impoverishment** and how **roughening
   noise** after resampling prevents the cloud from collapsing.
5. Interpret an **effective sample size (ESS)** curve and a position-error curve
   to judge filter health.


## Setup

Run the following cell first. It imports the libraries used by the simulation
and the live visualization.


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Circle
from IPython.display import display, clear_output

# Consistent, readable plot styling
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["grid.alpha"] = 0.4


## How a Particle Filter Works

A particle filter represents the belief over the robot's pose
$\mathbf{x} = [x, y, \theta]$ with a set of $N$ **particles**
$\{\mathbf{x}^{(1)}, \dots, \mathbf{x}^{(N)}\}$, each carrying a weight $w^{(i)}$.
The density of particles in a region *is* the probability the robot is there —
so the cloud can take **any shape**, not just a Gaussian.

Every time step runs four stages.

### 1. Predict (motion)

Push every particle forward using the robot's control $[v, \omega]$ over a time
step $\Delta t$, adding random noise so the particles spread the way our
uncertain odometry would:

$$
\begin{aligned}
x^{(i)} &\mathrel{+}= (v + \varepsilon_v)\,\Delta t\,\cos\theta^{(i)} \\
y^{(i)} &\mathrel{+}= (v + \varepsilon_v)\,\Delta t\,\sin\theta^{(i)} \\
\theta^{(i)} &\mathrel{+}= (\omega + \varepsilon_\omega)\,\Delta t
\end{aligned}
\qquad \varepsilon_v \sim \mathcal N(0,\sigma_v^2),\;\; \varepsilon_\omega \sim \mathcal N(0,\sigma_\omega^2)
$$

### 2. Weight (measurement update)

Score each particle by how well the readings it *would* produce match the
**actual** measurement $\mathbf{z}$. With range readings to $K$ landmarks
$\mathbf{L}_k$, the predicted range from particle $i$ is
$\hat z^{(i)}_k = \lVert \mathbf{p}^{(i)} - \mathbf{L}_k \rVert$, and the weight
is the Gaussian measurement likelihood

$$
w^{(i)} \;\propto\; \prod_{k=1}^{K} \exp\!\left(-\frac{\big(z_k - \hat z^{(i)}_k\big)^2}{2\,\sigma_z^2}\right).
$$

Particles that "explain" the measurement well get large weights; those that
don't get tiny ones. Weights are then normalized so they sum to 1.

### 3. Resample

Draw a **new** set of $N$ particles from the old set, choosing each with
probability proportional to its weight. High-weight particles get copied many
times; low-weight particles usually vanish. After resampling, all weights reset
to $1/N$ — the *information* now lives in **where** the particles are, not in the
weights.

### 4. Roughen (jitter the survivors)

Resampling copies winners **exactly**, so a few poses get duplicated dozens of
times — the cloud loses diversity (**sample impoverishment**). To fix this we
add a little Gaussian noise to every particle after resampling:

$$
\mathbf{x}^{(i)} \mathrel{+}= \boldsymbol{\eta}^{(i)}, \qquad
\eta_{x}, \eta_{y} \sim \mathcal N(0, \sigma_{\text{pos}}^2), \quad
\eta_{\theta} \sim \mathcal N(0, \sigma_{\theta}^2).
$$

This "roughening" keeps the swarm slightly spread so it can track the robot and
recover from surprises — the direct analogue of process noise in a Kalman
filter. Too little and the cloud collapses to a dead point; too much and the
estimate gets jittery.

### Getting the estimate

At any moment the pose estimate is the **weighted mean** of the particles
(for position), and the health of the filter is summarized by the
**effective sample size**

$$\text{ESS} = \frac{1}{\sum_i \big(w^{(i)}\big)^2}\,,$$

which ranges from $1$ (all weight on one particle — bad) up to $N$ (weights
even — healthy).


## The Three Knobs You Will Explore

This lab is built around the three design choices that most shape how a particle
filter behaves. You will implement each one and then experiment with it.

| Knob | Function you implement | What it controls | What to watch |
|---|---|---|---|
| **Number of particles** `N` | (a parameter you set) | How finely the cloud can represent the belief | Too few → the filter loses the robot; more → smoother but slower |
| **Initial distribution** | `initialize_particles` | Where the swarm starts before any data | `"uniform"` for a lost robot, `"gaussian"` for a rough guess, `"single_point"` to see collapse |
| **Roughening noise** | `roughen` | How much the survivors jitter each step | Too little → cloud collapses (impoverishment); too much → jittery estimate |

You will also implement `resample`, the step that turns weights into a new set
of particles, because roughening only makes sense once you understand what it is
repairing.

Implement the three functions in the next cells, then run the **Simulation
Engine** and the **experiment** cell to see them in action.


## Task 1 — Implement `initialize_particles()`

Create the starting cloud of `num_particles` particles. Each particle is a pose
`[x, y, theta]`, so the returned array has shape `(num_particles, 3)`. Support
three `mode` values:

| `mode` | Meaning | How to draw each column |
|---|---|---|
| `"uniform"` | Robot is **completely lost** — spread over the whole arena | `x, y ~ Uniform(0, arena_size)`, `theta ~ Uniform(-π, π)` |
| `"gaussian"` | Robot pose is **roughly known** — a blob around a guess | `x, y ~ Normal(center, spread)`, `theta ~ Normal(center_θ, 0.5)` |
| `"single_point"` | All particles at **one exact pose** (deliberately degenerate) | every particle `= init_center` |

Parameters:

- `num_particles` — number of particles `N`.
- `mode` — one of the three strings above.
- `arena_size` — the arena is a square from `0` to `arena_size` (meters).
- `init_center` — length-3 `[x, y, theta]` used by `"gaussian"` and `"single_point"`.
- `init_spread` — standard deviation (meters) of the Gaussian blob's `x` and `y`.
- `rng` — a `numpy.random.Generator`; **use it** for all randomness so results
  are reproducible.

### Hints

- Preallocate `particles = np.zeros((num_particles, 3))` and fill the columns.
- `rng.uniform(low, high, num_particles)` and
  `rng.normal(mean, std, num_particles)` each return a length-`N` array.
- For `"single_point"`, `particles[:] = init_center` broadcasts the pose to all rows.
- Return the `(num_particles, 3)` array.


In [ ]:
def initialize_particles(num_particles, mode, arena_size, init_center, init_spread, rng):
    """
    Create the initial cloud of particles.

    Parameters
    ----------
    num_particles : int
        Number of particles N.
    mode : {"uniform", "gaussian", "single_point"}
        How to draw the initial poses (see the task description).
    arena_size : float
        The arena is the square [0, arena_size] x [0, arena_size] (meters).
    init_center : array-like, shape (3,)
        Reference pose [x, y, theta] used by "gaussian" and "single_point".
    init_spread : float
        Std-dev (meters) of the x and y Gaussian blob for "gaussian".
    rng : numpy.random.Generator
        Use this for ALL randomness (rng.uniform, rng.normal).

    Returns
    -------
    numpy.ndarray, shape (num_particles, 3)
        The particle poses, one [x, y, theta] per row.
    """
    init_center = np.asarray(init_center, dtype=float)
    particles = np.zeros((num_particles, 3))

    # TODO: mode == "uniform"
    #   particles[:, 0] = rng.uniform(0, arena_size, num_particles)
    #   particles[:, 1] = rng.uniform(0, arena_size, num_particles)
    #   particles[:, 2] = rng.uniform(-np.pi, np.pi, num_particles)

    # TODO: mode == "gaussian"
    #   particles[:, 0] = rng.normal(init_center[0], init_spread, num_particles)
    #   particles[:, 1] = rng.normal(init_center[1], init_spread, num_particles)
    #   particles[:, 2] = rng.normal(init_center[2], 0.5, num_particles)

    # TODO: mode == "single_point"
    #   particles[:] = init_center     # every particle starts at the exact same pose

    # TODO: return particles

    pass  # Replace this line with your implementation


## Task 2 — Implement `resample()`

Given the current particles and their **normalized** weights, draw a new set of
`N` particles by selecting each existing particle with probability equal to its
weight. High-weight particles get duplicated; low-weight ones tend to disappear.

You will use **systematic resampling**, the standard low-variance method:

1. Build the cumulative sum of the weights: `cumsum = np.cumsum(weights)`.
2. Pick one random start offset in `[0, 1/N)`:
   `start = rng.random() / N`.
3. Lay down `N` equally spaced "pointers": `positions = start + arange(N) / N`.
4. Walk the pointers up the cumulative sum, recording which particle each
   pointer lands in. Those indices are your chosen particles.

Return the **new array of particle poses** (shape `(N, 3)`) — a copy, so later
roughening does not modify the originals.

### Hints

- `N = len(particles)`.
- A clean way to find the landing indices in one shot:
  `indices = np.searchsorted(cumsum, positions)`.
- Then `new_particles = particles[indices].copy()`.
- `np.clip(indices, 0, N - 1)` guards against a floating-point pointer landing
  exactly at `1.0`.


In [ ]:
def resample(particles, weights, rng):
    """
    Draw a new set of particles in proportion to their weights
    (systematic / low-variance resampling).

    Parameters
    ----------
    particles : ndarray, shape (N, 3)
        Current particle poses.
    weights : ndarray, shape (N,)
        Normalized weights (sum to 1).
    rng : numpy.random.Generator
        Use this for the single random start offset.

    Returns
    -------
    ndarray, shape (N, 3)
        The resampled particle poses (a copy). Weights are NOT returned;
        after resampling every particle is considered equally likely (1/N).
    """
    N = len(particles)

    # TODO: build the cumulative distribution of the weights
    #   cumsum = np.cumsum(weights)

    # TODO: pick a random start in [0, 1/N) and lay down N evenly spaced pointers
    #   start = rng.random() / N
    #   positions = start + np.arange(N) / N

    # TODO: find which particle each pointer lands in
    #   indices = np.searchsorted(cumsum, positions)
    #   indices = np.clip(indices, 0, N - 1)

    # TODO: return the selected particles as a copy
    #   return particles[indices].copy()

    pass  # Replace this line with your implementation


## Task 3 — Implement `roughen()`

Resampling copies winning particles **exactly**, so right after it the cloud
contains many identical duplicates. `roughen()` sprinkles a little Gaussian
noise on every particle so the swarm regains diversity and can keep tracking the
robot. This is the **"noise added to the good particles after each iteration"**
that makes or breaks a particle filter.

Add independent Gaussian noise to each column:

$$
x \mathrel{+}= \mathcal N(0, \sigma_{\text{pos}}^2),\quad
y \mathrel{+}= \mathcal N(0, \sigma_{\text{pos}}^2),\quad
\theta \mathrel{+}= \mathcal N(0, \sigma_{\theta}^2)
$$

Parameters:

- `particles` — shape `(N, 3)` array of poses.
- `roughening` — a 2-tuple `(pos_sigma, theta_sigma)`: the position noise std
  (meters) for `x`/`y`, and the heading noise std (radians) for `theta`.
- `rng` — the `numpy.random.Generator` to draw the noise from.

### Hints

- Work on a copy: `particles = particles.copy()`.
- `pos_sigma, theta_sigma = roughening`.
- Add `rng.normal(0, pos_sigma, N)` to columns 0 and 1, and
  `rng.normal(0, theta_sigma, N)` to column 2.
- If a sigma is `0`, `rng.normal(0, 0, N)` simply returns zeros — no special
  case needed.
- Return the noised array.


In [ ]:
def roughen(particles, roughening, rng):
    """
    Add Gaussian jitter to every particle after resampling to restore diversity
    (prevents sample impoverishment / particle collapse).

    Parameters
    ----------
    particles : ndarray, shape (N, 3)
        Resampled particle poses.
    roughening : (float, float)
        (pos_sigma, theta_sigma):
          pos_sigma   -- std-dev (meters) of the noise added to x and y
          theta_sigma -- std-dev (radians) of the noise added to theta
    rng : numpy.random.Generator
        Use this to draw the noise.

    Returns
    -------
    ndarray, shape (N, 3)
        The jittered particle poses (a copy).
    """
    particles = particles.copy()
    N = len(particles)
    pos_sigma, theta_sigma = roughening

    # TODO: add Gaussian noise to each column
    #   particles[:, 0] += rng.normal(0.0, pos_sigma, N)
    #   particles[:, 1] += rng.normal(0.0, pos_sigma, N)
    #   particles[:, 2] += rng.normal(0.0, theta_sigma, N)

    # TODO: return particles

    pass  # Replace this line with your implementation


## Simulation Engine

Run this cell after implementing the three functions above. It defines the
world (arena size, landmark beacons, noise levels), the scenario generator, the
**motion** and **measurement** models, the particle-filter loop that calls
**your** `initialize_particles`, `resample`, and `roughen`, and the animated
dashboard. **You do not need to modify any code in this cell.**

All distances are in **meters**.


In [ ]:
# ── World constants (meters) ───────────────────────────────────────────────────
ARENA_SIZE = 10.0                                   # square arena [0, 10] x [0, 10]
LANDMARKS = np.array([[1.0, 1.0], [9.0, 1.0],       # beacon positions on the walls
                      [9.0, 9.0], [1.0, 9.0]])
DT = 0.5                                             # time step (s)
MOTION_NOISE = (0.10, 0.05)   # particle predict noise: (v std m/s, omega std rad/s)
MEAS_NOISE = 0.25             # range measurement noise std (m)

# Default true starting pose of the robot [x, y, theta]
TRUE_START = np.array([5.0, 2.0, 0.0])


def wrap_angle(a):
    """Wrap angle(s) to (-pi, pi]."""
    return (np.asarray(a) + np.pi) % (2 * np.pi) - np.pi


# ── Motion model ───────────────────────────────────────────────────────────────

def _move_one(pose, control, dt, noise, rng):
    """Advance a single pose by one control step (used to drive the true robot)."""
    v, omega = control
    v = v + rng.normal(0.0, noise[0])
    omega = omega + rng.normal(0.0, noise[1])
    x, y, th = pose
    x = x + v * dt * np.cos(th)
    y = y + v * dt * np.sin(th)
    th = wrap_angle(th + omega * dt)
    return np.array([x, y, th])


def _predict_particles(particles, control, dt, noise, rng):
    """Push every particle forward by the control with per-particle motion noise."""
    v, omega = control
    N = len(particles)
    v_n = v + rng.normal(0.0, noise[0], N)
    omega_n = omega + rng.normal(0.0, noise[1], N)
    out = particles.copy()
    th = particles[:, 2]
    out[:, 0] = particles[:, 0] + v_n * dt * np.cos(th)
    out[:, 1] = particles[:, 1] + v_n * dt * np.sin(th)
    out[:, 2] = wrap_angle(th + omega_n * dt)
    return out


# ── Measurement model ──────────────────────────────────────────────────────────

def _measurement_likelihood(particles, z, landmarks, sigma):
    """
    Gaussian likelihood of the range measurements z for every particle.
    Returns an (N,) array of un-normalized weights (numerically stabilized).
    """
    px = particles[:, 0][:, None]
    py = particles[:, 1][:, None]
    pred = np.sqrt((px - landmarks[:, 0]) ** 2 + (py - landmarks[:, 1]) ** 2)  # (N, K)
    diff = pred - z[None, :]
    log_w = -0.5 * np.sum((diff / sigma) ** 2, axis=1)
    log_w -= log_w.max()          # stabilize before exponentiating
    return np.exp(log_w)


def _estimate(particles, weights):
    """Weighted-mean pose estimate (circular mean for the heading)."""
    x = np.average(particles[:, 0], weights=weights)
    y = np.average(particles[:, 1], weights=weights)
    s = np.average(np.sin(particles[:, 2]), weights=weights)
    c = np.average(np.cos(particles[:, 2]), weights=weights)
    return np.array([x, y, np.arctan2(s, c)])


# ── Scenario generator ─────────────────────────────────────────────────────────

def generate_scenario(num_steps=55, v=1.0, turn_rate=1.0 / 3.0, seed=0):
    """
    Drive the true robot on a gentle loop inside the arena and record, for each
    step, the control it was given and the noisy landmark ranges it measured.

    Returns a dict with:
        controls     : list of (v, omega) tuples, length num_steps
        measurements : list of (K,) range arrays, length num_steps
        true_poses   : (num_steps + 1, 3) array of true poses (index 0 = start)
    """
    rng = np.random.default_rng(seed)
    pose = TRUE_START.copy()
    true_poses = [pose.copy()]
    controls, measurements = [], []

    for _ in range(num_steps):
        control = (v, turn_rate)
        pose = _move_one(pose, control, DT, (0.03, 0.02), rng)  # small true process noise
        # keep the robot comfortably inside the arena
        pose[0] = np.clip(pose[0], 0.5, ARENA_SIZE - 0.5)
        pose[1] = np.clip(pose[1], 0.5, ARENA_SIZE - 0.5)
        true_poses.append(pose.copy())

        ranges = np.linalg.norm(pose[:2] - LANDMARKS, axis=1)
        z = ranges + rng.normal(0.0, MEAS_NOISE, len(LANDMARKS))
        controls.append(control)
        measurements.append(z)

    return dict(controls=controls,
                measurements=measurements,
                true_poses=np.array(true_poses))


# ── The particle filter loop (calls YOUR three functions) ──────────────────────

def run_particle_filter(scenario, num_particles=500, init_mode="gaussian",
                        init_center=(5.0, 2.0, 0.0), init_spread=1.0,
                        roughening=(0.10, 0.05), seed=0):
    """
    Run Monte Carlo Localization on a scenario using the student's
    initialize_particles(), resample(), and roughen().

    Returns a list (one entry per time step) of dicts:
        {particles, weights, est, ess}
    recorded AFTER weighting but BEFORE resampling, so the visualization can
    show the weighted cloud. history[t] corresponds to true_poses[t + 1].
    """
    rng = np.random.default_rng(seed)
    controls = scenario["controls"]
    measurements = scenario["measurements"]

    particles = np.asarray(
        initialize_particles(num_particles, init_mode, ARENA_SIZE,
                             init_center, init_spread, rng),
        dtype=float,
    )
    weights = np.full(num_particles, 1.0 / num_particles)

    history = []
    for t in range(len(controls)):
        # 1. Predict
        particles = _predict_particles(particles, controls[t], DT, MOTION_NOISE, rng)
        # 2. Weight
        w = _measurement_likelihood(particles, measurements[t], LANDMARKS, MEAS_NOISE)
        w = w * weights
        total = w.sum()
        weights = (np.full(num_particles, 1.0 / num_particles)
                   if (total <= 0 or not np.isfinite(total)) else w / total)
        # record for visualization
        est = _estimate(particles, weights)
        ess = 1.0 / np.sum(weights ** 2)
        history.append(dict(particles=particles.copy(), weights=weights.copy(),
                            est=est, ess=float(ess)))
        # 3. Resample  +  4. Roughen  (student code)
        particles = np.asarray(resample(particles, weights, rng), dtype=float)
        particles = np.asarray(roughen(particles, roughening, rng), dtype=float)
        weights = np.full(num_particles, 1.0 / num_particles)

    return history


# ── Visualization ──────────────────────────────────────────────────────────────

def _draw_arena(ax):
    ax.clear()
    ax.add_patch(plt.Rectangle((0, 0), ARENA_SIZE, ARENA_SIZE,
                               fill=False, edgecolor="black", linewidth=1.2))
    ax.scatter(LANDMARKS[:, 0], LANDMARKS[:, 1], marker="s", s=110,
               color="darkorange", edgecolor="black", zorder=6, label="Landmarks")
    ax.set_xlim(-0.5, ARENA_SIZE + 0.5)
    ax.set_ylim(-0.5, ARENA_SIZE + 0.5)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")


def _heading_arrow(ax, pose, color, length=0.9):
    ax.arrow(pose[0], pose[1], length * np.cos(pose[2]), length * np.sin(pose[2]),
             head_width=0.25, head_length=0.25, fc=color, ec=color, zorder=8)


def run_and_visualize(scenario, animate=True, fps=8, **pf_kwargs):
    """
    Run the particle filter on a scenario and animate the swarm converging.

    Left panel: the arena with landmarks, the particle cloud (shaded by weight),
    the true robot (green star + heading), and the estimate (red X + heading).
    Right panels: position error over time and effective sample size (ESS).
    """
    true_poses = scenario["true_poses"]

    try:
        history = run_particle_filter(scenario, **pf_kwargs)
    except Exception as exc:  # noqa: BLE001
        print(f"⚠ The particle filter raised an error: {exc}")
        print("  Make sure initialize_particles(), resample(), and roughen() are "
              "implemented and their cells have been run.")
        return

    errors = np.array([np.linalg.norm(h["est"][:2] - true_poses[t + 1][:2])
                       for t, h in enumerate(history)])
    ess = np.array([h["ess"] for h in history])
    N = len(history[0]["particles"])

    fig = plt.figure(figsize=(13, 6))
    gs = gridspec.GridSpec(2, 2, width_ratios=[1.35, 1.0], height_ratios=[1, 1])
    ax_arena = fig.add_subplot(gs[:, 0])
    ax_err = fig.add_subplot(gs[0, 1])
    ax_ess = fig.add_subplot(gs[1, 1])

    def _draw_side(k):
        ax_err.clear()
        ax_err.plot(range(k + 1), errors[:k + 1], "-o", color="crimson", markersize=3)
        ax_err.set_xlim(0, len(history))
        ax_err.set_ylim(0, max(errors.max() * 1.1, 0.5))
        ax_err.set_xlabel("time step")
        ax_err.set_ylabel("position error (m)")
        ax_err.set_title("Estimate error vs. true pose", fontsize=10)
        ax_err.grid(True, linestyle="--", alpha=0.4)

        ax_ess.clear()
        ax_ess.plot(range(k + 1), ess[:k + 1], "-o", color="steelblue", markersize=3)
        ax_ess.axhline(N, color="gray", linestyle=":", linewidth=1, label=f"N = {N}")
        ax_ess.axhline(N / 2, color="green", linestyle="--", linewidth=1,
                       label="N/2 (healthy)")
        ax_ess.set_xlim(0, len(history))
        ax_ess.set_ylim(0, N * 1.05)
        ax_ess.set_xlabel("time step")
        ax_ess.set_ylabel("effective sample size")
        ax_ess.set_title("Effective sample size (ESS)", fontsize=10)
        ax_ess.legend(fontsize=8, loc="lower right")
        ax_ess.grid(True, linestyle="--", alpha=0.4)

    def _draw_frame(k):
        h = history[k]
        p, w = h["particles"], h["weights"]
        _draw_arena(ax_arena)
        wn = w / w.max() if w.max() > 0 else w
        ax_arena.scatter(p[:, 0], p[:, 1], s=6 + 45 * wn, c=wn, cmap="viridis",
                         alpha=0.45, zorder=3)
        # true trajectory so far + current true pose
        ax_arena.plot(true_poses[:k + 2, 0], true_poses[:k + 2, 1],
                      color="green", linewidth=1.0, alpha=0.5, zorder=4)
        true_now = true_poses[k + 1]
        ax_arena.scatter([true_now[0]], [true_now[1]], marker="*", s=220,
                         color="green", edgecolor="black", zorder=9,
                         label="True robot")
        _heading_arrow(ax_arena, true_now, "green")
        # estimate
        est = h["est"]
        ax_arena.scatter([est[0]], [est[1]], marker="X", s=90, color="crimson",
                         edgecolor="black", zorder=9, label="Estimate")
        _heading_arrow(ax_arena, est, "crimson")
        ax_arena.set_title(f"Monte Carlo Localization — step {k + 1}/{len(history)}"
                           f"   (error {errors[k]:.2f} m)",
                           fontsize=11, weight="bold")
        ax_arena.legend(loc="upper left", fontsize=8)
        _draw_side(k)

    if animate:
        for k in range(len(history)):
            _draw_frame(k)
            clear_output(wait=True)
            display(fig)
            time.sleep(1.0 / fps)
    else:
        _draw_frame(len(history) - 1)

    clear_output(wait=True)
    display(fig)
    plt.close(fig)

    settled = errors[len(errors) // 2:].mean()
    print("── Filter summary ────────────────────────────")
    print(f"  Particles              : {N}")
    print(f"  Final position error   : {errors[-1]:.3f} m")
    print(f"  Mean error (2nd half)  : {settled:.3f} m")
    print(f"  Final ESS              : {ess[-1]:.1f} / {N}")
    print("──────────────────────────────────────────────")


## Try Your Filter

Run the cell below after implementing the three functions and running the
Simulation Engine cell. It animates the particle cloud converging onto the true
robot, with the position-error and ESS curves updating alongside.

Change the parameters and re-run to explore the three knobs:

| Parameter | Try | Effect |
|---|---|---|
| `num_particles` | `30`, `200`, `1000` | Too few loses the robot; more is smoother but slower |
| `init_mode` | `"uniform"`, `"gaussian"`, `"single_point"` | A lost robot vs. a rough guess vs. a collapsed cloud |
| `init_spread` | `0.5`, `2.0`, `5.0` | Size of the Gaussian starting blob (for `"gaussian"`) |
| `roughening` | `(0.0, 0.0)`, `(0.1, 0.05)`, `(0.6, 0.4)` | No jitter → collapse; too much → jittery estimate |
| `seed` | any integer | Different random draws (same true path) |

> Tip: set `animate=False` for an instant final-frame result while sweeping
> parameters, then turn it back on to watch the convergence.


In [ ]:
# ── Experiment parameters ──────────────────────────────────────────────────────
num_particles = 500                 # number of particles N
init_mode     = "gaussian"          # "uniform" | "gaussian" | "single_point"
init_center   = (5.0, 2.0, 0.0)     # [x, y, theta] guess for gaussian/single_point
init_spread   = 1.0                 # std-dev (m) of the gaussian starting blob
roughening    = (0.10, 0.05)        # (position sigma m, heading sigma rad) jitter
seed          = 0                   # random seed for the filter

scenario = generate_scenario(seed=0)   # the true robot path (keep seed fixed to compare)

run_and_visualize(
    scenario,
    animate=True,
    fps=8,
    num_particles=num_particles,
    init_mode=init_mode,
    init_center=init_center,
    init_spread=init_spread,
    roughening=roughening,
    seed=seed,
)


## Guided Experiments

Work through these three experiments — they map directly onto the three knobs.
Record what you observe; you will use it in the reflection questions.

**Experiment A — Number of particles.**
In the *Try Your Filter* cell, set `init_mode="uniform"` (a fully lost robot) and
run the filter with `num_particles` = `20`, `100`, `500`, `2000`. Watch how many
particles it takes before the swarm reliably finds and tracks the robot, and
notice how the animation slows as `N` grows.

**Experiment B — Initial distribution.**
Keep `num_particles=500` and compare the three `init_mode` values:
`"uniform"` (lost), `"gaussian"` with `init_center=(5.0, 2.0, 0.0)` (a good
guess), and `"gaussian"` with a *wrong* guess like `init_center=(8.0, 8.0, 0.0)`.
Then try `"single_point"` with `roughening=(0.0, 0.0)` and watch the cloud fail
to spread.

**Experiment C — Roughening noise.**
Keep `init_mode="gaussian"`, `num_particles=500`, and sweep `roughening` through
`(0.0, 0.0)`, `(0.1, 0.05)`, and `(0.6, 0.4)`. Watch the ESS curve and the cloud:
zero roughening collapses the swarm to a point (and the error creeps up when the
robot maneuvers), while too much roughening keeps the estimate jittery.

The cell below quantifies **Experiment A** automatically: it runs the filter (no
animation) for a range of particle counts and plots the settled position error.
Run it once your three functions work.


In [ ]:
def settled_error(scenario, **pf_kwargs):
    """Mean position error over the second half of the run (no animation)."""
    history = run_particle_filter(scenario, **pf_kwargs)
    true_poses = scenario["true_poses"]
    errs = np.array([np.linalg.norm(h["est"][:2] - true_poses[t + 1][:2])
                     for t, h in enumerate(history)])
    return errs[len(errs) // 2:].mean()


scenario = generate_scenario(seed=0)
particle_counts = [10, 20, 50, 100, 200, 500, 1000, 2000]

# Average over a few seeds so the curve is not dominated by one lucky/unlucky run
errors_vs_N = []
for N in particle_counts:
    runs = [settled_error(scenario, num_particles=N, init_mode="uniform",
                          roughening=(0.10, 0.05), seed=s) for s in range(5)]
    errors_vs_N.append(np.mean(runs))

plt.figure(figsize=(7, 4.5))
plt.plot(particle_counts, errors_vs_N, "-o", color="crimson")
plt.xscale("log")
plt.xlabel("number of particles N (log scale)")
plt.ylabel("settled position error (m)")
plt.title("Experiment A: more particles → better localization (with diminishing returns)")
plt.grid(True, which="both", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

print("  N      settled error (m)")
for N, e in zip(particle_counts, errors_vs_N):
    print(f"  {N:<6d} {e:.3f}")


## Self-Checks

Run this cell to automatically verify your three functions. It checks that
`initialize_particles` produces the right shapes and distributions, that
`resample` favors high-weight particles, that `roughen` injects noise of the
requested scale, and that the whole filter localizes the robot to within a
reasonable error.


In [ ]:
def _report(name, ok, detail=""):
    print(f"{'PASS' if ok else 'FAIL'}: {name}" + (f"  ({detail})" if detail else ""))
    return ok

_results = []

# ── initialize_particles ───────────────────────────────────────────────────────
rng = np.random.default_rng(0)
center = np.array([5.0, 2.0, 0.3])

try:
    P = np.asarray(initialize_particles(5000, "uniform", ARENA_SIZE, center, 1.0, rng), float)
    ok = (P.shape == (5000, 3)
          and P[:, 0].min() >= 0 and P[:, 0].max() <= ARENA_SIZE
          and P[:, 1].min() >= 0 and P[:, 1].max() <= ARENA_SIZE
          and P[:, 2].min() >= -np.pi - 1e-6 and P[:, 2].max() <= np.pi + 1e-6
          and P[:, 0].std() > 2.0)
    _results.append(_report("initialize uniform: shape, bounds, spread", ok,
                            f"x std {P[:, 0].std():.2f}"))
except Exception as exc:  # noqa: BLE001
    _results.append(_report("initialize uniform", False, f"error: {exc}"))

try:
    P = np.asarray(initialize_particles(5000, "gaussian", ARENA_SIZE, center, 1.0, rng), float)
    ok = (P.shape == (5000, 3)
          and abs(P[:, 0].mean() - center[0]) < 0.15
          and abs(P[:, 1].mean() - center[1]) < 0.15
          and abs(P[:, 0].std() - 1.0) < 0.2)
    _results.append(_report("initialize gaussian: centered blob, correct spread", ok,
                            f"mean ({P[:, 0].mean():.2f}, {P[:, 1].mean():.2f}), x std {P[:, 0].std():.2f}"))
except Exception as exc:  # noqa: BLE001
    _results.append(_report("initialize gaussian", False, f"error: {exc}"))

try:
    P = np.asarray(initialize_particles(100, "single_point", ARENA_SIZE, center, 1.0, rng), float)
    ok = P.shape == (100, 3) and np.allclose(P, center)
    _results.append(_report("initialize single_point: all identical to center", ok))
except Exception as exc:  # noqa: BLE001
    _results.append(_report("initialize single_point", False, f"error: {exc}"))

# ── resample ───────────────────────────────────────────────────────────────────
try:
    parts = np.arange(30).reshape(-1, 1) * np.ones((1, 3))   # 30 distinct particles
    w = np.zeros(30); w[7] = 1.0                              # all weight on particle 7
    R = np.asarray(resample(parts, w, rng), float)
    ok = R.shape == (30, 3) and np.allclose(R, parts[7])
    _results.append(_report("resample: all-weight-on-one duplicates that particle", ok))
except Exception as exc:  # noqa: BLE001
    _results.append(_report("resample (one-hot)", False, f"error: {exc}"))

try:
    parts = np.arange(1000).reshape(-1, 1) * np.ones((1, 3))
    w = np.zeros(1000); w[:100] = 1.0 / 100                   # only first 100 have weight
    R = np.asarray(resample(parts, w, rng), float)
    ok = R.shape == (1000, 3) and R[:, 0].max() <= 99         # zero-weight particles never chosen
    _results.append(_report("resample: zero-weight particles are dropped", ok,
                            f"max index kept {int(R[:, 0].max())}"))
except Exception as exc:  # noqa: BLE001
    _results.append(_report("resample (drop zeros)", False, f"error: {exc}"))

# ── roughen ────────────────────────────────────────────────────────────────────
try:
    parts = np.zeros((20000, 3))
    Rn = np.asarray(roughen(parts, (0.3, 0.1), rng), float)
    ok = (Rn.shape == (20000, 3)
          and abs(Rn[:, 0].std() - 0.3) < 0.03
          and abs(Rn[:, 2].std() - 0.1) < 0.02
          and abs(Rn[:, 0].mean()) < 0.02)
    _results.append(_report("roughen: adds noise of the requested scale", ok,
                            f"pos std {Rn[:, 0].std():.3f}, theta std {Rn[:, 2].std():.3f}"))
except Exception as exc:  # noqa: BLE001
    _results.append(_report("roughen (scale)", False, f"error: {exc}"))

try:
    parts = np.ones((500, 3)) * 3.0
    Rn = np.asarray(roughen(parts, (0.0, 0.0), rng), float)
    ok = np.allclose(Rn, 3.0)
    _results.append(_report("roughen: zero sigma leaves particles unchanged", ok))
except Exception as exc:  # noqa: BLE001
    _results.append(_report("roughen (zero sigma)", False, f"error: {exc}"))

# ── end-to-end localization ────────────────────────────────────────────────────
scenario = generate_scenario(seed=0)
try:
    err = settled_error(scenario, num_particles=800, init_mode="gaussian",
                        init_center=(5.0, 2.0, 0.0), init_spread=1.0,
                        roughening=(0.10, 0.05), seed=1)
    ok = err < 0.8
    _results.append(_report("end-to-end: gaussian start tracks the robot", ok,
                            f"settled error {err:.3f} m"))
except Exception as exc:  # noqa: BLE001
    _results.append(_report("end-to-end (gaussian)", False, f"error: {exc}"))

try:
    err = settled_error(scenario, num_particles=1500, init_mode="uniform",
                        roughening=(0.10, 0.05), seed=2)
    ok = err < 1.0
    _results.append(_report("end-to-end: lost robot (uniform) still localizes", ok,
                            f"settled error {err:.3f} m"))
except Exception as exc:  # noqa: BLE001
    _results.append(_report("end-to-end (uniform)", False, f"error: {exc}"))

print()
if all(_results):
    print("All tests passed! Great work.")
else:
    print("Some tests failed. Re-read the task for each function and check the hints.")


## Reflection Questions

Answer these in a markdown cell (or on the Canvas page) using what you observed
in the guided experiments.

1. **Number of particles.** From Experiment A, roughly how many particles were
   needed before the filter reliably localized the *lost* (uniform) robot? Why
   do more particles help, and why do the returns diminish? What is the cost of
   adding particles?

2. **Initial distribution.** When would you choose a `"uniform"` initialization
   over a `"gaussian"` one, and vice versa? What happened when you gave the
   Gaussian a *wrong* `init_center`, and could the filter still recover? Why?

3. **The single point.** With `init_mode="single_point"` and
   `roughening=(0.0, 0.0)`, describe what happened to the cloud and the estimate.
   Which one mechanism (motion noise, resampling, or roughening) is missing that
   would let the swarm spread out?

4. **Roughening noise.** Explain, in terms of the ESS curve and the particle
   cloud, why *too little* roughening hurts and why *too much* also hurts. What
   is a good rule of thumb for choosing the roughening scale relative to the
   motion and measurement noise?

5. **Particle filter vs. least squares.** Compared with the GPS least-squares
   solver from Lab 5, name one situation where a particle filter clearly wins
   and one where the least-squares approach is the better choice.


## How to Submit

Follow the instructions on the Canvas assignment page. You will typically submit:

- your completed `initialize_particles`, `resample`, and `roughen` functions
  (with the **Self-Checks** cell passing), and
- written answers to the reflection questions above, including the particle
  counts, initial distributions, and roughening settings you tried and the
  settled position error each produced.
